In [1]:
import io
import os
import pandas as pd
from azure.storage.filedatalake import DataLakeServiceClient
from dotenv import load_dotenv
from pathlib import Path
import numpy as np

In [2]:
path = "/home/scooperlosses/DB_proj/complex_ETL/scripts/transformation/data/raw_data.csv"

df = pd.read_csv(path, dtype=str)

In [20]:
# To repair type atributes for data...

info_df = pd.DataFrame({
    'Dtype': df.dtypes,
    'Non-Null Count': df.notna().sum(),
    'Null count': df.isna().sum()
})

with pd.option_context('display.max_rows', 50):
    display(info_df)

,Dtype,Non-Null Count,Null count
DATE_TIME_IDDOC_x36,str,287552,0
SalesDocument_Unique_IDx36,str,287552,0
SalesDocument_Number,object,287552,0
SalesDocument_LineNo,int64,287552,0
SalesDocument_Date,str,287552,0
SalesManager_Login_IDx36,str,287552,0
SalesManager_Login,str,287552,0
SalesManager_IDx36,str,287552,0
SalesManager_Code,int64,287552,0
SalesManager_Name,str,287552,0


In [147]:
# Strip
# Выбираем только те колонки, к которым реально применить .str.strip()
string_cols = df.select_dtypes(include=['object', 'string']).columns
for col in string_cols:
    df[col] = df[col].astype(str).str.strip()


# Cleaning DATE_TIME_IDDOC_x36
sample = '202605017HII4G'
normal_len = len(sample)
df["DATE_TIME_IDDOC_x36"] = df["DATE_TIME_IDDOC_x36"].astype(str).str[:normal_len].str.strip()


# Cleaning BarCode
df["BarCode_EAN13"] = df["BarCode_EAN13"].replace("", pd.NA)

mask_invalid = df["BarCode_EAN13"].astype(str).str.len() != 13;
df.loc[mask_invalid, "BarCode_EAN13"] = pd.NA

df = df.dropna(subset=["BarCode_EAN13"])


# Cleaning DiscountCardNo
df["DiscountCardNo"] = df["DiscountCardNo"].replace('', 'NO_CARD')


# Cleaning ProdParent0_Code
df["ProdParent0_Code"] = df["ProdParent0_Code"].replace("Incalzire autonoma si apa", pd.NA)
df["ProdParent0_Code"] = df["ProdParent0_Code"].replace("+-", pd.NA)
df['ProdParent0_Code'] = df["ProdParent0_Code"].fillna('Unknown')
df[df["ProdParent0_Code"].isin(["Incalzire autonoma si apa", "+-"]) & df["ProdParent0_Code"].notna()].shape[0]


# Cleaning UnitMeasure_Code
df["UnitMeasure_Code"] = df["UnitMeasure_Code"].replace('Buc', '1')
mask_numeric = df["UnitMeasure_Code"].astype(str).str.fullmatch(r'\d+')

cleaned = df.loc[mask_numeric, "UnitMeasure_Code"].astype(str).str.lstrip('0')
cleaned = cleaned.replace('', '0')

df.loc[mask_numeric, "UnitMeasure_Code"] = cleaned

# Cleaning VAT
df["VAT"] = df["VAT"].astype(str).str.replace('%', '', regex=False)


# SalesManager_Login ...
# Customer_Name
df["Customer_Name"] = df["Customer_Name"].astype(str).str.replace('CARD ', '', regex=False) 
garbage_pattern = r'\s+(\d{2}\.\d{2}\.\d{4}|\d+%|[A-Za-z]\d+-[A-Za-z\d]+|[A-Za-z]\d+)\s*$'

df["Customer_Name"] = df["Customer_Name"].astype(str).str.replace(garbage_pattern, '', regex=True)

# Cleaning UnitMeasure_Name
unit_mapping = {
    'buc': 'piece',    
    'set': 'pack',     
    'pac': 'pack', 
    'cut': 'pack',
    'sac': 'weight',   
    'kg': 'weight',
    'm': 'length',     
    'm2': 'area',      
    'rul': 'length',   
    'per': 'piece',    
    'unknown': 'unknown'
}

# Сначала приводим к нижнему регистру, чтобы маппинг сработал для 'Buc' и 'buc' одинаково, и используем replace вместо map
df["UnitMeasure_Name"] = df["UnitMeasure_Name"].fillna("unknown").astype(str).str.lower()
df["UnitMeasure_Name"] = df["UnitMeasure_Name"].replace(unit_mapping)

# Prod_Name
df['Prod_Name']= df['Prod_Name'].astype(str).str.lower()
df['Prod_Name'] = df['Prod_Name'].str.replace(r'\s*\([^)]*\)', '', regex=True)
df['Prod_Name'] = df['Prod_Name'].str.replace(r'\s+([a-z]*\d{4,7}|\d{4,7})\b\s*$', '', regex=True)
df['Prod_Name'] = df['Prod_Name'].str.strip()

# ProdParent1_Name
df['ProdParent1_Name'] = (df['ProdParent1_Name'].astype(str)
                          .str.lower()
                          .str.replace(r'[\s]*,[\s]*', ' / ', regex=True)
                          .str.replace(r'\s+', ' ', regex=True)
                          .str.strip())
                          

# ProdParent0_Name
df['ProdParent0_Name'] = (df['ProdParent0_Name'].fillna('unknown').astype(str)
                          .str.lower())      

# SalesDate_WeekDay
# Исправил опечатки в названиях дней (Wednesday, Thursday)
week_day={
    '1':'Monday',
    '2': 'Tuesday',
    '3': 'Wednesday',
    '4': 'Thursday',
    '5': 'Friday',
    '6': 'Saturday',
    '7': 'Sunday'
}

df['SalesDate_WeekDay'] = (df['SalesDate_WeekDay'].astype(str).replace(week_day)
                           .replace('0', 'unknown_day'))

# SalesDate_MonthNum
month_map = {
    '1': 'January',
    '2': 'February',
    '3': 'March',
    '4': 'April',
    '5': 'May',
    '6': 'June',
    '7': 'July',
    '8': 'August',
    '9': 'September',
    '10': 'October',
    '11': 'November',
    '12': 'December'
}
df['SalesDate_MonthNum'] = (df['SalesDate_MonthNum'].astype(str).replace(month_map)
                            .replace('0', 'Unknown_month'))


# WH_Code
df['WH_Name'] = df['WH_Name'].astype(str).str.replace(r'^_+', '', regex=True).str.strip()
is_numeric_string = df['WH_Name'].str.contains(r'^\d+\.\d+$|^\d+$', regex=True, na=False)
df['WH_Name'] = np.where(is_numeric_string, 'Unknown_Warehouse', df['WH_Name'])
df['WH_Name'] = df['WH_Name'].replace(['nan', 'None'], 'Unknown_Warehouse').fillna('Unknown_Warehouse')

#
df['Quantity'] = pd.to_numeric(df['Quantity'], errors='coerce').fillna(0.0)

date_cols = ['SalesDocument_Date', 'SalesDate_WeekStart', 'SalesDate_WeekEnd']
for col in date_cols:
    df[col] = pd.to_datetime(df[col], errors='coerce')

# 2. Переводим Дробные числа (float)
float_cols = [
    'Quantity', 'Price', 'TotalAmount', 'VATamount', 'VAT', 
    'checksum_TotalAmount', 'checksum_VATAmount', 'UnitMeasure_Koef', '_Weight', '_Volume'
]
for col in float_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0.0)

# 3. Переводим Целые числа (int)
# Убрал 'SalesDate_MonthNum' из int_cols, так как выше мы перевели её в текстовые названия месяцев ('January' и т.д.)
int_cols = ['SalesDocument_LineNo', 'SalesDate_Hour', 'SalesDate_WeekNum', 'SalesDate_MonthDayNum']
for col in int_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0).astype(int)

# 4. Переводим повторяющийся текст в Категории (для оптимизации памяти)
# Добавил 'SalesDate_MonthNum' сюда, так как теперь это текстовые категории месяцев
cat_cols = ['SalesDate_WeekDay', 'SalesDate_MonthNum', 'SalesManager_Name', 'WH_Name', 'ProdParent0_Name', 'ProdParent1_Name', 'UnitMeasure_Name']
for col in cat_cols:
    df[col] = df[col].astype('category')

df[df.columns[20:40]].info()

<class 'pandas.DataFrame'>
Index: 148630 entries, 0 to 287550
Data columns (total 20 columns):
 #   Column                Non-Null Count   Dtype   
---  ------                --------------   -----   
 0   BarCode_EAN13         148630 non-null  str     
 1   Prod_Unique_IDx36     148630 non-null  str     
 2   Prod_Code             148630 non-null  str     
 3   Prod_Name             148630 non-null  str     
 4   Prod_FullName         148630 non-null  str     
 5   ProdParent1_Code      148630 non-null  str     
 6   ProdParent1_Name      148630 non-null  category
 7   ProdParent0_Code      148630 non-null  str     
 8   ProdParent0_Name      148630 non-null  category
 9   WH_Code               148630 non-null  str     
 10  WH_Name               148630 non-null  category
 11  Quantity              148630 non-null  float64 
 12  Price                 148630 non-null  float64 
 13  TotalAmount           148630 non-null  float64 
 14  VATamount             148630 non-null  float64 
 15 